# Event Weights

Calculate concurrency, average uniqueness, return attribution, time decay, and normalized sample weights from the labeled AAPL events. Development and holdout are processed independently while preserving the established 65-column weighted-event schema.


## Process the Data


In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing.event_weights import (
    apply_time_decay,
    compute_average_uniqueness_weights,
    compute_return_attribution_weights,
    count_concurrent_events,
)

period = "2025-01-01_2025-12-31"
event_dir = PROJECT_ROOT / "data/research_data/events"
event_path = event_dir / f"aapl_news_primary_model_{period}.parquet"
partition_path = event_dir / f"aapl_news_labeled_split_{period}.parquet"
dollar_path = PROJECT_ROOT / f"data/research_data/market/features/aapl_dollar_bar_{period}.parquet"
weighted_path = event_dir / f"aapl_news_modeling_weighted_{period}.parquet"

events = pd.read_parquet(event_path).sort_values("event_start", ignore_index=True)
partition_manifest = pd.read_parquet(partition_path).sort_values("event_start", ignore_index=True)
dollar_bars = pd.read_parquet(dollar_path).sort_values("end").drop_duplicates("end", keep="last")
events["event_start"] = pd.to_datetime(events["event_start"], utc=True)
events["event_end"] = pd.to_datetime(events["event_end"], utc=True)
partition_manifest["event_start"] = pd.to_datetime(partition_manifest["event_start"], utc=True)
close = dollar_bars.set_index("end")["close"].astype(float)


In [ ]:
weight_tables = []
for partition in ["development", "holdout"]:
    partition_starts = partition_manifest.loc[partition_manifest["partition"].eq(partition), "event_start"]
    partition_events = events[events["event_start"].isin(partition_starts)].set_index("event_start")
    information_sets = partition_events["event_end"]
    concurrency = count_concurrent_events(close.index, information_sets, information_sets.index)
    uniqueness = compute_average_uniqueness_weights(information_sets, concurrency, information_sets.index)
    return_attribution = compute_return_attribution_weights(information_sets, concurrency, close, information_sets.index)
    positive_floor = return_attribution[return_attribution.gt(0)].min()
    if pd.isna(positive_floor):
        raise ValueError(f"{partition} return-attribution weights are all zero.")
    base_weight = return_attribution.clip(lower=positive_floor)
    time_decay = apply_time_decay(base_weight, clf_last_w=0.50)
    sample_weight = base_weight * time_decay
    sample_weight *= len(sample_weight) / sample_weight.sum()
    weight_tables.append(
        pd.DataFrame(
            {
                "average_uniqueness_weight": uniqueness,
                "return_attribution_weight": return_attribution,
                "time_decay_weight": time_decay,
                "sample_weight": sample_weight,
            }
        ).rename_axis("event_start").reset_index()
    )

weight_table = pd.concat(weight_tables, ignore_index=True).sort_values("event_start", ignore_index=True)
weighted_events = events.merge(weight_table, on="event_start", how="left", validate="one_to_one")
weight_columns = ["average_uniqueness_weight", "return_attribution_weight", "time_decay_weight", "sample_weight"]

assert weighted_events.shape[1] == 65
assert weighted_events["return_attribution_weight"].ge(0).all()
assert weighted_events[["average_uniqueness_weight", "time_decay_weight", "sample_weight"]].gt(0).all().all()
for partition in ["development", "holdout"]:
    starts = partition_manifest.loc[partition_manifest["partition"].eq(partition), "event_start"]
    partition_weights = weighted_events.loc[weighted_events["event_start"].isin(starts), "sample_weight"]
    assert abs(partition_weights.mean() - 1.0) < 1e-12

weighted_events.to_parquet(weighted_path, index=False)
print(weighted_path)


## Take a Quick Look at the Data Structure


In [ ]:
development_starts = partition_manifest.loc[partition_manifest["partition"].eq("development"), "event_start"]
development_data = weighted_events[weighted_events["event_start"].isin(development_starts)]
development_data.head()

In [ ]:
development_data.info()

In [ ]:
development_data["direction_label"].value_counts()

In [ ]:
development_data[weight_columns].describe()

In [ ]:
development_data[weight_columns].hist(figsize=(12, 8), bins=30)